In [1]:
import pickle
from pathlib import Path
from google.colab import drive

# Persistent storage
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/dl-assignment3")
CACHE = ROOT / "cache" / "v1"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS = ROOT / "results"

CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# Load preprocessed data
with open(CACHE / "prepared.pkl", "rb") as f:
    prepared = pickle.load(f)

print("Environment ready!")
print("Word vocabulary:", len(prepared["word"]["vocab"]))
print("BPE vocabulary:", prepared["bpe"]["vocab_size"])

Mounted at /content/drive
Environment ready!
Word vocabulary: 179708
BPE vocabulary: 16000


## Актуальний запуск — довший word regularization study

**На живому Runtime:** виконайте секції **1 → 4 → 5 (обидві комірки) → 6 → 7 → 9 → 10**. У секції 9 встановіть RUN_BATCH = True. prepared лишається в RAM; стартову комірку та data definitions повторювати не потрібно. Імпорти додають EarlyStopping, модель — metadata для легких snapshots, training/evaluation — нову логіку.

**Після Runtime reset:** стартова комірка → секції 1–7 → секція 8 (login) → секції 9–10. Встановлення залежностей потрібне лише за їх відсутності. Для resume вкажіть BATCH_ID та RESUME_BATCH = True, зберігши початкову кількість епох.

План: три word BiLSTM на всьому train з одним seed, lr=0.001 і cosine. Контроль: Adam/dropout=0.3; потім dropout=0.4; потім AdamW/dropout=0.4/weight_decay=0.01. Кожній дозволено до 10 епох, мінімум 3, early stopping patience=2 і min_delta=0.001. Це довший режим; за швидкістю попередніх word-моделей досягнення всіх лімітів може зайняти приблизно 1–1.5 години, а early stopping може завершити раніше.

Зберігаються top-2 snapshots за val_f1 і найкращий за val_loss. Checkpoint вибирається за threshold-tuned validation F1; labeling оцінюється лише після вибору. Окремі last/recovery checkpoints містять повний optimizer/scheduler/RNG/callback state для resume. Early stopping вважається успішним завершенням моделі.

Старі word/BPE моделі не перенавчаються: їхні CV/LB результати включено до порівняння. Відомі LB: word=0.55325, BPE=0.55182; нові LB залишаються порожніми. Comp_metric, split, кеш v1, BIO loss та назви W&B метрик збережено. Labeling і synthetic не входять у train; додаткових моделей поза GRU/LSTM немає.

### 1. Імпорти та константи

In [2]:
%pip install -q "pytorch-lightning>=2.5,<2.7" "wandb>=0.19,<1"

In [3]:
# Виконайте цю команду вручну лише якщо бракує залежностей, потім повторіть imports.
# %pip install -q "pytorch-lightning>=2.5,<2.7" "wandb>=0.19,<1"
import os, re, json, string, random, hashlib, uuid, platform
from datetime import datetime, timezone
from getpass import getpass
from importlib.metadata import version, PackageNotFoundError
from typing import List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from pytorch_lightning.loggers import CSVLogger, WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

PAD, UNK, IGNORE = 0, 1, -100
PIPELINE_VERSION = 'rnn-v1-team-legacy'
WANDB_PROJECT = 'dl-3'
WANDB_ENTITY = 'yrachynska-kyiv-school-of-economics'
THRESHOLDS = np.arange(0.10, 0.91, 0.05)
# Після ручної зміни семантики моделі/метрики змініть PIPELINE_VERSION.
print('torch:', torch.__version__, '| Lightning:', pl.__version__, '| GPU:', torch.cuda.is_available())

torch: 2.14.1+cu130 | Lightning: 2.6.6 | GPU: True


### 2. Competition metric і decoding

Legacy regex та поведінка множин відповідають колегам. POST_RULES передаються явно.

In [4]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for t, pr in zip(y_true, y_pred):
        t = set([hard_processing(s) for s in t])
        pr = set([hard_processing(s) for s in pr])
        tp += len(t & pr); fp += len(pr - t); fn += len(t - pr); p += len(t)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(f1, 8)

EMOJI = re.compile('[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE0F]')

def clean_span(s, rules):
    if 'newline' in rules:
        s = s.split('\n')[0]
    if 'emoji' in rules:
        s = EMOJI.sub('', s)
    s = s.strip()
    if 'strip' in rules:
        s = s.strip(' \t,.;:!?-–—"«»“”()')
    if 'letters' in rules and not re.search(r'[^\W\d_]', s):
        return None
    if 'upper' in rules and not any(w[0].isupper() for w in re.findall(r'\w+', s)):
        return None
    return s or None

def decode(text, offsets, probs, thr, post_rules=()):
    is_loc = (1 - probs[:, 0]) > thr
    result, start, end = [], None, None
    for t, (t_start, t_end) in enumerate(offsets[:len(probs)]):
        new_loc = is_loc[t] and (probs[t, 1] >= probs[t, 2] or t == 0 or not is_loc[t - 1])
        if new_loc:
            if start is not None:
                result.append(text[start:end])
            start, end = t_start, t_end
        elif is_loc[t] and start is not None:
            end = t_end
        else:
            if start is not None:
                result.append(text[start:end])
            start, end = None, None
    if start is not None:
        result.append(text[start:end])
    if post_rules:
        result = [c for c in (clean_span(x, post_rules) for x in result) if c]
    return result

def score(probs_list, texts, offsets_list, golds, thr, post_rules=()):
    assert len(probs_list) == len(texts) == len(offsets_list) == len(golds)
    preds = [decode(t, o, p, thr, post_rules) for t, o, p in zip(texts, offsets_list, probs_list)]
    return comp_metric(golds, preds)

### 3. Дані, batching і повнодокументний inference

Кеш перевикористовується. Loss використовує max_len; competition F1 — усі chunks документа.

In [5]:
def json_hash(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                     separators=(',', ':')).encode('utf8')).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf8')
    os.replace(tmp, path)


_PROVENANCE_CACHE = {}

def inspect_prepared(kind):
    # Перевірка та fingerprint виконуються один раз на kind у поточному kernel.
    cache_key = (id(prepared), kind)
    if cache_key in _PROVENANCE_CACHE:
        return _PROVENANCE_CACHE[cache_key]
    data = prepared[kind]
    vocab_size = len(data['vocab']) if kind == 'word' else data['vocab_size']
    counts, fingerprints = {}, {}
    for split in ('big', 'val', 'lab', 'test'):
        d = data[split]
        assert len(d['ids']) == len(d['offsets']), split
        if split != 'test':
            assert len(d['labels']) == len(d['ids']), split
        counts[split] = len(d['ids'])
        digest = hashlib.sha256()
        for i, (ids, offsets) in enumerate(zip(d['ids'], d['offsets'])):
            assert len(ids) == len(offsets), (split, i)
            assert not ids or (min(ids) >= 0 and max(ids) < vocab_size), (split, i)
            row = [ids, offsets]
            if split != 'test':
                labels = d['labels'][i]
                assert len(labels) == len(ids) and set(labels) <= {0, 1, 2}, (split, i)
                row.append(labels)
            if split != 'big':
                text = prepared[split + '_text'][i]
                assert all(0 <= a <= b <= len(text) for a, b in offsets), (split, i)
                row.append(text)
                if split != 'test':
                    row.append(d['locs'][i])
            digest.update(json.dumps(row, ensure_ascii=False, separators=(',', ':')).encode('utf8'))
            digest.update(b'\n')
        if split != 'big':
            assert len(prepared[split + '_text']) == counts[split], split
            if split != 'test':
                assert len(d['locs']) == counts[split], split
        fingerprints[split] = digest.hexdigest()
    tokenizer = data['vocab'] if kind == 'word' else data.get('tokenizer_json')
    if tokenizer is None:
        tokenizer = (CACHE / 'bpe.json').read_text(encoding='utf8')
    result = dict(counts=counts, fingerprints=fingerprints, vocab_size=int(vocab_size),
                  tokenizer_sha256=json_hash(tokenizer))
    _PROVENANCE_CACHE[cache_key] = result
    print('Cache:', kind, counts, '| validation SHA256:', fingerprints['val'][:16])
    return result


class NERDataset(Dataset):
    def __init__(self, ids, labels, max_len):
        self.ids, self.labels, self.max_len = ids, labels, max_len

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return (torch.tensor(self.ids[i][:self.max_len], dtype=torch.long),
                torch.tensor(self.labels[i][:self.max_len], dtype=torch.long))


def collate(batch):
    ids, labels = zip(*batch)
    return dict(input_ids=pad_sequence(ids, batch_first=True, padding_value=PAD),
                labels=pad_sequence(labels, batch_first=True, padding_value=IGNORE),
                lengths=torch.tensor([len(x) for x in ids], dtype=torch.long))


class EpochShuffleSampler(Sampler):
    # Відтворюваний порядок кожної епохи, незалежний від dropout/global RNG.
    def __init__(self, dataset, seed):
        self.dataset, self.seed, self.epoch = dataset, seed, 0

    def __len__(self):
        return len(self.dataset)

    def __iter__(self):
        generator = torch.Generator().manual_seed(self.seed + self.epoch)
        return iter(torch.randperm(len(self), generator=generator).tolist())


def make_loaders(cfg):
    data = prepared[cfg['kind']]
    idx = random.Random(cfg['seed']).sample(
        range(len(data['big']['ids'])), min(cfg['n_train'], len(data['big']['ids'])))
    ids = [data['big']['ids'][i] for i in idx]
    labels = [data['big']['labels'][i] for i in idx]
    if cfg['use_lab']:
        ids += data['lab']['ids'] * cfg['use_lab']
        labels += data['lab']['labels'] * cfg['use_lab']
    keep = [i for i, x in enumerate(ids) if len(x)]
    train_ds = NERDataset([ids[i] for i in keep], [labels[i] for i in keep], cfg['max_len'])
    val_keep = [i for i, x in enumerate(data['val']['ids']) if len(x)]
    val_ds = NERDataset([data['val']['ids'][i] for i in val_keep],
                        [data['val']['labels'][i] for i in val_keep], cfg['max_len'])
    if not len(train_ds) or not len(val_ds):
        raise ValueError('Порожній train або validation для loss.')
    sampler = EpochShuffleSampler(train_ds, cfg['seed'])
    loader_kwargs = dict(collate_fn=collate, num_workers=cfg['num_workers'])
    # Worker/base seeds не повинні зрушувати global RNG, який використовує dropout.
    train_dl = DataLoader(train_ds, batch_size=cfg['batch_size'], sampler=sampler,
                          generator=torch.Generator().manual_seed(cfg['seed']), **loader_kwargs)
    val_dl = DataLoader(val_ds, batch_size=cfg['eval_batch_size'], shuffle=False,
                        generator=torch.Generator().manual_seed(cfg['seed'] + 1), **loader_kwargs)
    return train_dl, val_dl, dict(train_examples=len(train_ds), train_indices_sha256=json_hash(idx),
                                  val_loss_examples=len(val_ds))


@torch.inference_mode()
def predict_probs(model, ids_list, max_len, batch_size):
    was_training = model.training
    model.eval()
    try:
        chunks = [(d, s, ids[s:s + max_len]) for d, ids in enumerate(ids_list)
                  for s in range(0, max(len(ids), 1), max_len)]
        pieces = {}
        for b in range(0, len(chunks), batch_size):
            batch = chunks[b:b + batch_size]
            x = pad_sequence([torch.tensor(c[2] if len(c[2]) else [PAD], dtype=torch.long)
                              for c in batch], batch_first=True, padding_value=PAD)
            lengths = torch.tensor([max(len(c[2]), 1) for c in batch], dtype=torch.long)
            probs = torch.softmax(model(x.to(model.device), lengths), dim=-1).float().cpu().numpy()
            for (d, s, ids), p in zip(batch, probs):
                pieces[(d, s)] = p[:len(ids)]
        return [np.concatenate([pieces[(d, s)] for s in range(0, len(ids), max_len)])
                if len(ids) else np.zeros((0, 3), dtype='float32')
                for d, ids in enumerate(ids_list)]
    finally:
        model.train(was_training)

### 4. Модель і checkpoint metadata — повторити

Параметр weight_decay явний. Легкі snapshots призначені для evaluation, повні last/recovery — для продовження тренування.

In [6]:
class NERModel(pl.LightningModule):
    def __init__(self, vocab_size, emb_dim=200, hidden=256, layers=1, bidirectional=True,
                 rnn='lstm', dropout=0.3, lr=2e-3, optimizer='adam', scheduler='onecycle',
                 loc_weight=2.0, emb_matrix=None, max_len=256, eval_batch_size=256, post_rules=(), weight_decay=0.01):
        super().__init__()
        self.save_hyperparameters(ignore=['emb_matrix'])
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        if emb_matrix is not None:
            with torch.no_grad():
                self.emb.weight.copy_(torch.as_tensor(emb_matrix))
        rnn_cls = {'lstm': nn.LSTM, 'gru': nn.GRU}[rnn]
        self.rnn = rnn_cls(emb_dim, hidden, num_layers=layers, batch_first=True,
                           bidirectional=bidirectional, dropout=dropout if layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden * (2 if bidirectional else 1), 3)
        self.register_buffer('class_weight', torch.tensor([1.0, loc_weight, loc_weight]))
        self.val_info = None

    def on_save_checkpoint(self, checkpoint):
        # This hook also runs for lightweight evaluation snapshots (callback hooks do not).
        if hasattr(self, 'run_metadata'):
            checkpoint['rnn_run'] = self.run_metadata

    def forward(self, ids, lengths):
        x = self.drop(self.emb(ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))
        return self.out(self.drop(out))

    def _loss(self, batch):
        logits = self(batch['input_ids'], batch['lengths'])
        return nn.functional.cross_entropy(logits.reshape(-1, 3), batch['labels'].reshape(-1),
                                           weight=self.class_weight, ignore_index=IGNORE)

    def training_step(self, batch, batch_idx):
        loss = self._loss(batch)
        self.log('train_loss', loss, on_step=True, on_epoch=False, prog_bar=True,
                 batch_size=len(batch['lengths']))
        return loss

    def validation_step(self, batch, batch_idx):
        self.log('val_loss', self._loss(batch), on_step=False, on_epoch=True, prog_bar=True,
                 batch_size=len(batch['lengths']))

    def on_before_optimizer_step(self, optimizer):
        norms = [p.grad.norm() ** 2 for p in self.parameters() if p.grad is not None]
        if norms:
            self.log('grad_norm', torch.sqrt(torch.stack(norms).sum()), on_step=True, on_epoch=False)

    def on_validation_epoch_end(self):
        if self.val_info is None or self.trainer.sanity_checking:
            return
        texts, offsets, golds, ids = self.val_info
        probs = predict_probs(self, ids, self.hparams.max_len, self.hparams.eval_batch_size)
        f1 = score(probs, texts, offsets, golds, 0.5, self.hparams.post_rules)
        self.log('val_f1', f1, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        if self.hparams.optimizer == 'adam':
            opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        elif self.hparams.optimizer == 'adamw':
            opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        else:
            opt = torch.optim.SGD(self.parameters(), lr=self.hparams.lr, momentum=0.9)
        steps = int(self.trainer.estimated_stepping_batches)
        if self.hparams.scheduler == 'onecycle':
            sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=self.hparams.lr, total_steps=steps)
        elif self.hparams.scheduler == 'cosine':
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        else:
            return opt
        return dict(optimizer=opt, lr_scheduler=dict(scheduler=sch, interval='step'))


class RunState(pl.Callback):
    def __init__(self, cfg, provenance, sampler, run_dir):
        self.cfg, self.provenance, self.sampler, self.run_dir = cfg, provenance, sampler, Path(run_dir)
        self.pending_rng = None

    def on_train_epoch_start(self, trainer, pl_module):
        self.sampler.epoch = trainer.current_epoch

    def on_save_checkpoint(self, trainer, pl_module, checkpoint):
        state = np.random.get_state()
        checkpoint['rnn_run'] = dict(config=self.cfg, provenance=self.provenance,
                                    pipeline_version=PIPELINE_VERSION)
        checkpoint['rnn_rng'] = dict(python=random.getstate(), torch=torch.get_rng_state(),
            cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            numpy=dict(name=state[0], keys=state[1].tolist(), pos=state[2],
                       has_gauss=state[3], cached_gaussian=state[4]))

    def on_load_checkpoint(self, trainer, pl_module, checkpoint):
        saved = checkpoint.get('rnn_run', {})
        if saved.get('config') != self.cfg or saved.get('provenance') != self.provenance:
            raise ValueError('Checkpoint не відповідає CONFIG або даним. Створіть новий run.')
        if saved.get('pipeline_version') != PIPELINE_VERSION:
            raise ValueError('Інша версія pipeline; створіть новий run.')
        self.pending_rng = checkpoint.get('rnn_rng')

    def on_train_start(self, trainer, pl_module):
        if self.pending_rng:
            state = self.pending_rng
            random.setstate(state['python'])
            torch.set_rng_state(state['torch'])
            if torch.cuda.is_available() and len(state['cuda']) == torch.cuda.device_count():
                torch.cuda.set_rng_state_all(state['cuda'])
            n = state['numpy']
            np.random.set_state((n['name'], np.asarray(n['keys'], dtype=np.uint32),
                                 n['pos'], n['has_gauss'], n['cached_gaussian']))
            self.pending_rng = None

    def on_exception(self, trainer, pl_module, exception):
        # Жорстке вимкнення VM цей hook не викличе; для нього є periodic/last.
        if trainer.global_step:
            try:
                trainer.save_checkpoint(self.run_dir / 'interrupted.ckpt')
            except Exception:
                print('Аварійне збереження не вдалося; використайте останній periodic/last checkpoint.')

### 5. W&B та training — повторити обидві code-комірки

Нові параметри early stopping і checkpoint policy логуються в config/summary. Звичайна зупинка через early stopping завершує run успішно та дозволяє перейти до evaluation.

In [7]:
def login_wandb():
    import warnings
    import wandb
    key = os.environ.get('WANDB_API_KEY')
    if not key:
        with warnings.catch_warnings():
            warnings.simplefilter('error', category=__import__('getpass').GetPassWarning)
            key = getpass('W&B API key (hidden): ')
    try:
        if not key or not wandb.login(key=key, relogin=True):
            raise RuntimeError('Login failed')
    except Exception:
        raise RuntimeError('W&B login не вдався. Перевірте секрет і доступ до командного проєкту.') from None
    finally:
        del key
    globals()['_WANDB_READY'] = True
    print('W&B login готовий; доступ до проєкту буде перевірено при створенні run.')


def validate_config(cfg):
    assert cfg['kind'] in ('word', 'bpe') and cfg['rnn'] in ('lstm', 'gru')
    assert cfg['optimizer'] in ('adam', 'adamw', 'sgd')
    assert cfg['scheduler'] in ('onecycle', 'cosine', 'none')
    for key in ('emb_dim', 'hidden', 'layers', 'batch_size', 'eval_batch_size', 'max_len',
                'n_train', 'epochs', 'checkpoint_every_n_steps'):
        assert isinstance(cfg[key], int) and cfg[key] > 0, key
    assert isinstance(cfg['use_lab'], int) and cfg['use_lab'] >= 0
    assert cfg['num_workers'] >= 0 and 0 <= cfg['dropout'] < 1
    assert cfg['lr'] > 0 and cfg['loc_weight'] > 0 and cfg['gradient_clip_val'] > 0
    assert set(cfg['post_rules']) <= {'newline', 'emoji', 'strip', 'letters', 'upper'}
    assert 0 <= cfg.get('min_epochs', 0) <= cfg['epochs']
    assert cfg.get('save_top_k', 1) >= 1
    assert cfg.get('weight_decay', 0.0) >= 0
    patience = cfg.get('early_stopping_patience')
    assert patience is None or (isinstance(patience, int) and patience >= 1)
    assert cfg.get('early_stopping_min_delta', 0.0) >= 0
    if cfg['use_pretrained']:
        assert cfg['kind'] == 'word' and prepared['word'].get('emb_matrix') is not None
        assert prepared['word']['emb_matrix'].shape == (len(prepared['word']['vocab']), cfg['emb_dim'])


def runtime_versions():
    result = dict(python=platform.python_version())
    for name in ('torch', 'pytorch-lightning', 'numpy', 'wandb'):
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = None
    return result


def run_paths(run_id):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', run_id):
        raise ValueError('RUN_ID може містити лише літери, цифри, _ та -.')
    return CHECKPOINTS / run_id, RESULTS / run_id


def load_owned_checkpoint(path):
    # Лише власні checkpoints із Drive; не використовуйте файли з невідомих джерел.
    return torch.load(path, map_location='cpu', weights_only=False)


def latest_checkpoint(run_dir):
    candidates = [run_dir / 'last.ckpt', run_dir / 'recovery' / 'last.ckpt',
                  run_dir / 'interrupted.ckpt']
    found = []
    for path in candidates:
        if path.is_file():
            checkpoint = load_owned_checkpoint(path)
            found.append((int(checkpoint['global_step']), path.stat().st_mtime, path))
            del checkpoint
    if not found:
        raise FileNotFoundError('Не знайдено last/recovery/interrupted checkpoint.')
    return max(found)[2]


def open_wandb(cfg, manifest, result_dir, resume):
    if not cfg['use_wandb']:
        return None
    import wandb
    if not globals().get('_WANDB_READY', False):
        raise RuntimeError('Спочатку виконайте login_wandb() у цьому Runtime.')
    if wandb.run is not None:
        raise RuntimeError('Є активний W&B run. Завершіть його через wandb.finish().')
    logger = WandbLogger(project=WANDB_PROJECT, entity=WANDB_ENTITY,
        name=manifest['name'], id=manifest['wandb_id'],
        resume='must' if resume and manifest.get('wandb_started') else 'never',
        save_dir=str(result_dir), log_model=False, mode='online')
    try:
        run = logger.experiment
        if run.entity != WANDB_ENTITY or run.project != WANDB_PROJECT:
            raise RuntimeError('Unexpected W&B destination')
        run.config.update(dict(**cfg, pipeline_version=PIPELINE_VERSION,
                               provenance=manifest['provenance'], versions=manifest['versions']))
        # Summary val_f1 вручну оновлюється після threshold tuning.
        run.summary.update(dict(history_val_f1_threshold=0.5,
                               summary_val_f1_policy='validation threshold sweep 0.10:0.90:0.05'))
        return logger
    except Exception:
        if wandb.run is not None:
            wandb.finish(exit_code=1)
        raise RuntimeError('Немає доступу до командного W&B або сталася мережева помилка. '
                           'Проєкт автоматично не змінювався.') from None

In [8]:
def train_experiment(config, run_id=None, resume=False):
    cfg = json.loads(json.dumps(config))
    validate_config(cfg)
    if not torch.cuda.is_available() and cfg['accelerator'] != 'cpu':
        raise RuntimeError('GPU не знайдено. Увімкніть Colab GPU; CPU дозволяється лише явно.')
    if cfg['use_wandb'] and not globals().get('_WANDB_READY', False):
        raise RuntimeError('Спочатку виконайте login_wandb() у цьому Runtime.')
    provenance = inspect_prepared(cfg['kind'])
    pl.seed_everything(cfg['seed'], workers=True)
    train_dl, val_dl, subset = make_loaders(cfg)
    provenance = dict(**provenance, **subset)
    if resume and not run_id:
        raise ValueError('Для resume потрібен RUN_ID.')
    if not run_id:
        run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
    run_dir, result_dir = run_paths(run_id)
    manifest_path = result_dir / 'run.json'
    ckpt_path = None
    if resume:
        manifest = json.loads(manifest_path.read_text(encoding='utf8'))
        if manifest['config'] != cfg or manifest['provenance'] != provenance:
            raise ValueError('CONFIG/дані змінилися. Для нового експерименту RESUME = False, RUN_ID = None.')
        if manifest['pipeline_version'] != PIPELINE_VERSION:
            raise ValueError('Версія pipeline змінилася; потрібен новий run.')
        ckpt_path = latest_checkpoint(run_dir)
    else:
        run_dir.mkdir(parents=True, exist_ok=False)
        result_dir.mkdir(parents=True, exist_ok=False)
        manifest = dict(run_id=run_id, name=cfg['name'] + '_' + run_id, wandb_id=uuid.uuid4().hex[:8],
                        wandb_started=False, config=cfg, provenance=provenance,
                        pipeline_version=PIPELINE_VERSION, versions=runtime_versions(), status='created')
        atomic_json(manifest_path, manifest)
    print('RUN_ID =', run_id, '| resume:', ckpt_path, '| train examples:', subset['train_examples'])
    wb = None
    completed = False
    try:
        wb = open_wandb(cfg, manifest, result_dir, resume)
        if wb is not None:
            manifest['wandb_started'] = True
        manifest['status'] = 'training'
        atomic_json(manifest_path, manifest)
        data = prepared[cfg['kind']]
        model_keys = ('emb_dim', 'hidden', 'layers', 'bidirectional', 'rnn', 'dropout', 'lr',
                      'optimizer', 'scheduler', 'loc_weight', 'max_len', 'eval_batch_size', 'post_rules')
        model = NERModel(provenance['vocab_size'], **{k: cfg[k] for k in model_keys},
                         emb_matrix=data.get('emb_matrix') if cfg['use_pretrained'] else None,
                         weight_decay=cfg.get('weight_decay', 0.01 if cfg['optimizer']=='adamw' else 0.0))
        model.run_metadata = dict(config=cfg, provenance=provenance, pipeline_version=PIPELINE_VERSION)
        model.val_info = (prepared['val_text'], data['val']['offsets'], data['val']['locs'], data['val']['ids'])
        best = ModelCheckpoint(dirpath=str(run_dir), filename='best-{epoch:02d}-{step}',
            monitor='val_f1', mode='max', save_top_k=cfg.get('save_top_k', 1), save_last=False,
            save_weights_only=cfg.get('evaluation_weights_only', False))
        loss_best = None
        if cfg.get('save_best_val_loss', False):
            loss_best = ModelCheckpoint(dirpath=str(run_dir / 'loss'), filename='min-loss-{epoch:02d}-{step}',
                monitor='val_loss', mode='min', save_top_k=1, save_last=False,
                save_weights_only=cfg.get('evaluation_weights_only', False))
        stopper = None
        if cfg.get('early_stopping_patience') is not None:
            stopper = EarlyStopping(monitor='val_f1', mode='max',
                patience=cfg['early_stopping_patience'], min_delta=cfg.get('early_stopping_min_delta', 0.0),
                check_on_train_epoch_end=True, verbose=True)
        # Окремий unmonitored callback: last оновлюється навіть без покращення F1.
        epoch_last = ModelCheckpoint(dirpath=str(run_dir), filename='last', monitor=None,
            every_n_epochs=1, save_on_train_epoch_end=True, save_top_k=1, save_last=False,
            enable_version_counter=False, save_weights_only=False)
        recovery = ModelCheckpoint(dirpath=str(run_dir / 'recovery'), filename='last', monitor=None,
            every_n_train_steps=cfg['checkpoint_every_n_steps'], save_on_train_epoch_end=False,
            save_top_k=1, save_last=False, enable_version_counter=False, save_weights_only=False)
        state = RunState(cfg, provenance, train_dl.sampler, run_dir)
        loggers = [CSVLogger(str(result_dir / 'logs'), name='csv')]
        if wb is not None:
            loggers.append(wb)
        callbacks = [state]
        if stopper is not None:
            callbacks.append(stopper)
        callbacks.append(best)
        if loss_best is not None:
            callbacks.append(loss_best)
        # Full optimizer/scheduler/RNG/callback state remains in last/recovery checkpoints.
        callbacks.extend([epoch_last, recovery])
        trainer = pl.Trainer(max_epochs=cfg['epochs'], min_epochs=cfg.get('min_epochs', 0),
            accelerator=cfg['accelerator'], devices=1,
            precision=cfg['precision'], gradient_clip_val=cfg['gradient_clip_val'],
            logger=loggers, callbacks=callbacks, log_every_n_steps=50,
            enable_model_summary=False, enable_progress_bar=False, num_sanity_val_steps=0)
        trainer.fit(model, train_dl, val_dl, ckpt_path=str(ckpt_path) if ckpt_path else None)
        early_stopped = bool(stopper is not None and stopper.stopped_epoch > 0 and trainer.should_stop)
        completed = trainer.current_epoch >= cfg['epochs'] or early_stopped
        reason = 'early_stopping' if early_stopped else ('max_epochs' if completed else 'interrupted')
        ranked = sorted(best.best_k_models, key=lambda p: float(best.best_k_models[p]), reverse=True)
        if loss_best is not None and loss_best.best_model_path:
            ranked.append(loss_best.best_model_path)
        manifest.update(status='trained' if completed else 'interrupted', stop_reason=reason,
                        epochs_completed=trainer.current_epoch, checkpoint_candidates=list(dict.fromkeys(ranked)),
                        best_checkpoint=best.best_model_path, global_step=trainer.global_step)
        print('Training finished:', reason, '| epochs:', trainer.current_epoch, '| step:', trainer.global_step)
        atomic_json(manifest_path, manifest)
        if wb is not None:
            wb.experiment.summary.update(dict(training_complete=completed, stop_reason=reason,
                                               epochs_completed=trainer.current_epoch, run_id=run_id,
                                               **{k: v for k, v in cfg.items() if k != 'name'}))
        return run_id
    finally:
        if wb is not None:
            import wandb
            wandb.finish(exit_code=0 if completed else 1)

### 6. Evaluation — повторити

Порівнює збережені checkpoints за threshold-tuned validation F1, а не за labeling. checkpoint_scores.csv містить результати всіх кандидатів. Однакові global_step не оцінюються двічі.

In [9]:
def best_checkpoint(run_dir, manifest):
    named = manifest.get('best_checkpoint')
    if named and Path(named).is_file():
        return Path(named)
    choices = list(run_dir.glob('best-*.ckpt'))
    if len(choices) != 1:
        raise FileNotFoundError('Best checkpoint не знайдений однозначно. Спочатку завершіть validation.')
    return choices[0]


def evaluation_checkpoints(run_dir, manifest):
    paths = manifest.get('checkpoint_candidates')
    if not paths:
        return [best_checkpoint(run_dir, manifest)]
    result = [Path(p) for p in dict.fromkeys(paths)]
    if not all(p.is_file() for p in result):
        raise FileNotFoundError('Один із відібраних checkpoints відсутній на Drive.')
    return result


def evaluation_model(path, cfg, provenance):
    checkpoint = load_owned_checkpoint(path)
    metadata = checkpoint.get('rnn_run', {})
    if metadata.get('config') != cfg or metadata.get('provenance') != provenance:
        raise ValueError('Checkpoint не відповідає CONFIG/даним manifest.')
    if metadata.get('pipeline_version') != PIPELINE_VERSION:
        raise ValueError('Checkpoint має іншу версію pipeline.')
    model = NERModel(**checkpoint['hyper_parameters'])
    model.load_state_dict(checkpoint['state_dict'])
    epoch, step = int(checkpoint['epoch']), int(checkpoint['global_step'])
    del checkpoint
    model.to('cuda' if torch.cuda.is_available() else 'cpu')
    return model, epoch, step


def evaluate_experiment(run_id, predict_test=False, log_wandb=True):
    import gc
    run_dir, result_dir = run_paths(run_id)
    manifest = json.loads((result_dir / 'run.json').read_text(encoding='utf8'))
    cfg = manifest['config']
    validate_config(cfg)
    if manifest['pipeline_version'] != PIPELINE_VERSION:
        raise ValueError('Версія pipeline відрізняється від навчання.')
    actual = inspect_prepared(cfg['kind'])
    for key in ('counts', 'fingerprints', 'vocab_size', 'tokenizer_sha256'):
        if actual[key] != manifest['provenance'][key]:
            raise ValueError('Кеш змінився після навчання: ' + key)
    data = prepared[cfg['kind']]
    best_result, best_val_probs = None, None
    rows, all_thresholds, seen_steps = [], [], set()
    # Only validation participates in checkpoint selection. Labeling/test run after this loop.
    for path in evaluation_checkpoints(run_dir, manifest):
        model, epoch, step = evaluation_model(path, cfg, manifest['provenance'])
        if step in seen_steps:
            del model
            gc.collect()
            continue
        seen_steps.add(step)
        val_probs = predict_probs(model, data['val']['ids'], cfg['max_len'], cfg['eval_batch_size'])
        del model
        gc.collect()
        scores = [score(val_probs, prepared['val_text'], data['val']['offsets'], data['val']['locs'],
                        float(t), cfg['post_rules']) for t in THRESHOLDS]
        chosen = int(np.argmax(scores))
        record = dict(checkpoint=str(path), checkpoint_epoch=epoch + 1, evaluation_global_step=step,
            thr=float(THRESHOLDS[chosen]), val_f1=scores[chosen],
            **{'val_f1@0.5': score(val_probs, prepared['val_text'], data['val']['offsets'],
                                  data['val']['locs'], 0.5, cfg['post_rules'])})
        rows.append(record)
        all_thresholds.extend(dict(checkpoint=str(path), threshold=round(float(t), 2), val_f1=v)
                              for t, v in zip(THRESHOLDS, scores))
        if best_result is None or record['val_f1'] > best_result['val_f1']:
            best_result, best_val_probs = record, val_probs
    if best_result is None:
        raise RuntimeError('Немає checkpoint для evaluation.')
    path = Path(best_result['checkpoint'])
    threshold = best_result['thr']
    model, _, _ = evaluation_model(path, cfg, manifest['provenance'])
    probs = dict(val=best_val_probs,
                 lab=predict_probs(model, data['lab']['ids'], cfg['max_len'], cfg['eval_batch_size']))
    if predict_test:
        probs['test'] = predict_probs(model, data['test']['ids'], cfg['max_len'], cfg['eval_batch_size'])
    del model
    gc.collect()
    row = dict(**best_result, run_id=run_id, name=manifest['name'], kind=cfg['kind'],
        lab_f1=None if cfg['use_lab'] else score(probs['lab'], prepared['lab_text'], data['lab']['offsets'],
                                               data['lab']['locs'], threshold, cfg['post_rules']),
        lab_in_train=cfg['use_lab'], checkpoints_evaluated=len(rows),
        checkpoint_selection_policy='validation threshold-tuned F1; first candidate wins ties')
    row['thr'] = round(threshold, 2)
    output = result_dir / ('eval_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:4])
    output.mkdir(parents=True, exist_ok=False)
    if predict_test:
        preds = [decode(t, o, p, threshold, cfg['post_rules'])
                 for t, o, p in zip(prepared['test_text'], data['test']['offsets'], probs['test'])]
        text_ids = data['test'].get('text_id')
        if text_ids is None or len(text_ids) != len(preds):
            raise ValueError('Потрібні справжні test text_id.')
        pd.DataFrame(dict(text_id=text_ids, locations=[json.dumps(x, ensure_ascii=False) for x in preds])).to_csv(
            output / 'submission.csv', index=False)
    pd.DataFrame(rows).to_csv(output / 'checkpoint_scores.csv', index=False)
    pd.DataFrame(all_thresholds).to_csv(output / 'thresholds_all_checkpoints.csv', index=False)
    pd.DataFrame([dict(threshold=r['threshold'], val_f1=r['val_f1']) for r in all_thresholds
                  if r['checkpoint']==str(path)]).to_csv(output / 'thresholds.csv', index=False)
    with open(output / 'predictions.pkl', 'wb') as f:
        pickle.dump(dict(probs=probs, thr=threshold, kind=cfg['kind'], post_rules=cfg['post_rules']), f)
    atomic_json(output / 'metrics.json', dict(**row, config=cfg, provenance=manifest['provenance']))
    if log_wandb and cfg['use_wandb']:
        wb = open_wandb(cfg, manifest, result_dir, resume=True)
        try:
            wb.experiment.summary.update({**{k:v for k,v in cfg.items() if k!='name'}, **row})
            wb.experiment.summary.update(dict(val_f1_fixed_0_5=row['val_f1@0.5'], val_f1_tuned=row['val_f1']))
        finally:
            import wandb
            wandb.finish()
    print('Evaluation:', output)
    print({k:row[k] for k in ('checkpoint_epoch','checkpoints_evaluated','val_f1@0.5','val_f1','lab_f1','thr')})
    return row, output

### 7. Автоматична черга та ансамблі — повторити

Три нові runs, потім validation-only вибір окремої моделі, парного або рівновагового ансамблю. Старі word/BPE враховуються лише як references; ймовірності різних токенізацій не змішуються.

In [10]:
def load_batch_json(path):
    return json.loads(Path(path).read_text(encoding='utf8'))


def batch_paths(batch_id):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', batch_id):
        raise ValueError('Некоректний BATCH_ID.')
    return RESULTS / 'batches' / batch_id


def read_evaluation(directory):
    directory = Path(directory)
    return load_batch_json(directory / 'metrics.json'), directory


def compare_full_models(state, directory):
    records = [load_batch_json(Path(t['evaluation_dir'])/'metrics.json') for t in state['tasks']]
    saved = []
    for task in state['tasks']:
        with open(Path(task['evaluation_dir'])/'predictions.pkl','rb') as f:
            saved.append(pickle.load(f))
    kind = records[0]['config']['kind']
    assert all(s['kind']==kind for s in saved)
    assert all(r['config']['kind']==kind and r['config']['use_lab']==0 for r in records)
    for key in ('fingerprints','tokenizer_sha256'):
        assert all(r['provenance'][key]==records[0]['provenance'][key] for r in records)
    data, count = prepared[kind], len(saved)
    for split in ('val','lab','test'):
        for member in saved:
            assert len(member['probs'][split])==len(data[split]['ids'])
            assert all(p.shape==(len(ids),3) for p,ids in zip(member['probs'][split],data[split]['ids']))
    weight_key = state['plan'].get('weight_key','alpha_lstm')
    vectors = state['plan'].get('member_weight_candidates')
    if vectors is None:
        assert count==2
        vectors = [[1.0,0.0],[0.0,1.0],[0.5,0.5]]
    assert all(len(v)==count and min(v)>=0 and np.isclose(sum(v),1.0) for v in vectors)
    def blend(split, weights):
        return [sum(w*member['probs'][split][i] for w,member in zip(weights,saved) if w)
                for i in range(len(data[split]['ids']))]
    candidates=[]
    for weights in vectors:
        val_probs=blend('val',weights)
        for rules in state['plan']['rule_sets']:
            for threshold in THRESHOLDS:
                value=score(val_probs,prepared['val_text'],data['val']['offsets'],data['val']['locs'],
                            float(threshold),rules)
                candidates.append(dict(member_weights=weights,post_rules=rules,thr=float(threshold),val_f1=value))
    chosen=max(candidates,key=lambda c:c['val_f1'])
    weights,rules,threshold=chosen['member_weights'],chosen['post_rules'],chosen['thr']
    probs={split:blend(split,weights) for split in ('val','lab','test')}
    weight_metadata={weight_key:weights if weight_key=='member_weights' else weights[0]}
    row=dict(name='mykyta_'+kind+'_selection_'+state['batch_id'],run_id=state['batch_id']+'_selection',
        kind=kind,**weight_metadata,post_rules=rules,members=[t['run_id'] for t in state['tasks']],
        thr=round(threshold,2),val_f1=chosen['val_f1'],
        **{'val_f1@0.5':score(probs['val'],prepared['val_text'],data['val']['offsets'],data['val']['locs'],0.5,rules)},
        lab_f1=score(probs['lab'],prepared['lab_text'],data['lab']['offsets'],data['lab']['locs'],threshold,rules),
        lab_in_train=0,selection_policy='validation only: member weights + post_rules + threshold')
    directory.mkdir(parents=True,exist_ok=True)
    pd.DataFrame([{**c,'member_weights':json.dumps(c['member_weights']),'post_rules':'+'.join(c['post_rules']),
                   'thr':round(c['thr'],2)} for c in candidates]).to_csv(directory/'validation_search.csv',index=False)
    preds=[decode(text,offsets,p,threshold,rules) for text,offsets,p in zip(
        prepared['test_text'],data['test']['offsets'],probs['test'])]
    ids=data['test'].get('text_id')
    if ids is None or len(ids)!=len(preds):
        raise ValueError('Не збігаються test text_id.')
    pd.DataFrame(dict(text_id=ids,locations=[json.dumps(p,ensure_ascii=False) for p in preds])).to_csv(
        directory/'submission.csv',index=False)
    with open(directory/'predictions.pkl','wb') as f:
        pickle.dump(dict(probs=probs,thr=threshold,kind=kind,post_rules=rules,**weight_metadata),f)
    atomic_json(directory/'metrics.json',row)
    return row


def log_reported_lb(reference_batch_id, score):
    # Лише фактичний score, отриманий користувачем на Kaggle; не прогноз score.
    import wandb
    if wandb.run is not None:
        raise RuntimeError('Спочатку завершіть активний W&B run.')
    reference = load_batch_json(batch_paths(reference_batch_id) / 'batch.json')
    logger = WandbLogger(project=WANDB_PROJECT, entity=WANDB_ENTITY,
        id=reference['selection_wandb_id'], resume='must', mode='online', log_model=False,
        save_dir=str(batch_paths(reference_batch_id)))
    try:
        run = logger.experiment
        if run.entity != WANDB_ENTITY or run.project != WANDB_PROJECT:
            raise RuntimeError('Unexpected W&B destination')
        run.summary.update(dict(LB=float(score), LB_source='User Kaggle submission screenshot, 2026-11-12'))
    finally:
        wandb.finish()


def run_experiment_batch(baseline_run_id, epochs=4, batch_id=None, resume=False,
                         study='word_lstm_gru', reference_batch_id=None, reference_lb=None, reported_lb_scores=None):
    import gc
    import time
    if 'evaluate_experiment' not in globals():
        raise RuntimeError('Спочатку виконайте комірку визначення evaluate_experiment вище.')
    if not baseline_run_id:
        raise ValueError('Вкажіть BASELINE_RUN_ID завершеного baseline.')
    _, baseline_dir = run_paths(baseline_run_id)
    baseline = load_batch_json(baseline_dir / 'run.json')
    if baseline.get('status') != 'trained':
        raise ValueError('Спочатку завершіть baseline training.')
    if baseline['pipeline_version'] != PIPELINE_VERSION:
        raise ValueError('Baseline має іншу версію pipeline.')
    base = baseline['config']
    if base['kind'] != 'word' or base['use_lab'] != 0:
        raise ValueError('Для цього batch потрібен word baseline без labeling у train.')
    if base['accelerator'] != 'cpu' and not torch.cuda.is_available():
        raise RuntimeError('Підключіть Colab GPU.')
    if base['use_wandb'] and not globals().get('_WANDB_READY', False):
        raise RuntimeError('Спочатку виконайте login_wandb().')
    assert isinstance(epochs, int) and epochs > 0
    if study not in ('word_lstm_gru', 'bpe_depth', 'word_regularization'):
        raise ValueError('Невідомий study.')
    kind = 'bpe' if study == 'bpe_depth' else 'word'
    word_provenance = inspect_prepared('word')
    current = word_provenance
    for key in ('counts', 'fingerprints', 'vocab_size', 'tokenizer_sha256'):
        if current[key] != baseline['provenance'][key]:
            raise ValueError('Дані відрізняються від baseline: ' + key)
    reference = None
    if study == 'bpe_depth':
        current = inspect_prepared('bpe')
        if current['counts'] != word_provenance['counts']:
            raise ValueError('Word/BPE мають різні розміри splits.')
        for split in ('val', 'lab'):
            if prepared['word'][split]['locs'] != prepared['bpe'][split]['locs']:
                raise ValueError('Word/BPE golds не узгоджені.')
        if prepared['word']['test']['text_id'] != prepared['bpe']['test']['text_id']:
            raise ValueError('Word/BPE test IDs не узгоджені.')
    if study in ('bpe_depth', 'word_regularization'):
        if reference_batch_id:
            reference = load_batch_json(batch_paths(reference_batch_id) / 'batch.json')
            if reference['status'] != 'finished' or reference['plan']['baseline_run_id'] != baseline_run_id:
                raise ValueError('Потрібен завершений reference batch із тим самим baseline.')
            if reference['plan']['provenance'] != word_provenance:
                raise ValueError('Reference batch має інші word-дані.')
        if reference_lb is not None:
            if reference is None or not 0 <= reference_lb <= 1:
                raise ValueError('Для відомого LB потрібен reference batch та score у [0, 1].')
    plan = dict(baseline_run_id=baseline_run_id, epochs=epochs, n_train=current['counts']['big'],
                rule_sets=[[], ['strip'], ['newline', 'emoji', 'letters', 'upper']],
                alpha_candidates=[1.0, 0.0, 0.5], pipeline_version=PIPELINE_VERSION,
                provenance=current)
    if study == 'bpe_depth':
        plan.update(study=study, weight_key='alpha_1layer', reference_batch_id=reference_batch_id,
                    reference_lb=reference_lb)
    if study == 'word_regularization':
        regularization = dict(lr=1e-3, scheduler='cosine', min_epochs=min(3,epochs),
            early_stopping_patience=2, early_stopping_min_delta=0.001,
            save_top_k=2, save_best_val_loss=True, evaluation_weights_only=True)
        reported_lb_scores = dict(reported_lb_scores or {})
        reported_references = []
        for reported_batch_id, value in reported_lb_scores.items():
            if not 0 <= value <= 1:
                raise ValueError('LB має бути в [0,1].')
            reported_state = load_batch_json(batch_paths(reported_batch_id)/'batch.json')
            if reported_state['status'] != 'finished':
                raise ValueError('Reported LB має посилатися на завершений batch.')
            selected_dir = batch_paths(reported_batch_id)/'selection'
            selected_row = load_batch_json(selected_dir/'metrics.json')
            reported_references.append(dict(batch_id=reported_batch_id, directory=str(selected_dir),
                run_id=selected_row['run_id'], kind=selected_row['kind'], LB=float(value)))
        plan.update(study=study, study_revision='word-regularization-v1', weight_key='member_weights',
            reference_batch_id=reference_batch_id, reference_lb=reference_lb,
            reported_lb_scores=reported_lb_scores, training_settings=regularization,
            variants=['word_control_d03_adam','word_d04_adam','word_d04_adamw'],
            member_weight_candidates=[[1,0,0],[0,1,0],[0,0,1],[0.5,0.5,0],[0.5,0,0.5],[0,0.5,0.5],
                                      [1/3,1/3,1/3]])
    if resume and not batch_id:
        raise ValueError('Для resume потрібен BATCH_ID.')
    if batch_id is None:
        batch_id = 'mykyta_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]
    directory = batch_paths(batch_id)
    ledger = directory / 'batch.json'
    if resume:
        state = load_batch_json(ledger)
        if state['plan'] != plan:
            raise ValueError('План або дані змінилися. Відновіть початкові налаштування batch.')
    else:
        directory.mkdir(parents=True, exist_ok=False)
        tasks = []
        variants = [('lstm', 1, 'lstm_full'), ('gru', 1, 'gru_full')]
        if study == 'bpe_depth':
            variants = [('lstm', 1, 'bpe_lstm_1layer'), ('lstm', 2, 'bpe_lstm_2layers')]
        if study == 'word_regularization':
            variants = [('lstm',1,label) for label in plan['variants']]
        for architecture, layers, label in variants:
            cfg = {**base, 'name': 'mykyta_' + label + '_' + str(epochs) + 'ep',
                   'kind': kind, 'rnn': architecture, 'layers': layers, 'use_pretrained': False,
                   'n_train': plan['n_train'], 'epochs': epochs, 'variant': label,
                   'use_lab': 0, 'post_rules': [], 'experiment_batch': batch_id, 'synth': 0}
            if study == 'word_regularization':
                cfg.update(plan['training_settings'])
                cfg.update(dropout=0.3 if label=='word_control_d03_adam' else 0.4,
                           optimizer='adamw' if label=='word_d04_adamw' else 'adam',
                           weight_decay=0.01 if label=='word_d04_adamw' else 0.0,
                           study_revision=plan['study_revision'])
            tasks.append(dict(config=cfg, run_id=None, attempts=[], evaluation_dir=None))
        state = dict(batch_id=batch_id, plan=plan, tasks=tasks, status='created',
                     baseline_evaluation=None, selection_logged=False,
                     selection_wandb_id=uuid.uuid4().hex[:8], selection_wandb_started=False,
                     elapsed_seconds=0.0)
        if reference is not None:
            state.update(baseline_evaluation=reference['baseline_evaluation'],
                         reference_selection=str(batch_paths(reference_batch_id) / 'selection'),
                         reference_lb_logged=False)
        if study == 'word_regularization':
            state.update(reported_references=reported_references, reported_lb_logged=[])
        atomic_json(ledger, state)
    print('BATCH_ID =', batch_id, '| відновлення:', resume, '| план:', epochs, 'епохи на модель')
    if state['status'] == 'finished':
        print('Batch уже завершений:', directory / 'comparison.csv')
        return batch_id, pd.read_csv(directory / 'comparison.csv')
    started = time.monotonic()
    state['status'] = 'running'
    atomic_json(ledger, state)
    try:
        if reference_lb is not None and base['use_wandb'] and not state.get('reference_lb_logged', False):
            log_reported_lb(reference_batch_id, reference_lb)
            state['reference_lb_logged'] = True
            atomic_json(ledger, state)
        for item in state.get('reported_references',[]):
            if base['use_wandb'] and item['batch_id'] not in state['reported_lb_logged']:
                log_reported_lb(item['batch_id'],item['LB'])
                state['reported_lb_logged'].append(item['batch_id'])
                atomic_json(ledger,state)
        total_stages=len(state['tasks'])+2
        if state['baseline_evaluation'] is None:
            print('1/'+str(total_stages)+': baseline evaluation')
            _, output = evaluate_experiment(baseline_run_id, predict_test=True)
            state['baseline_evaluation'] = str(output)
            atomic_json(ledger, state)
        for i, task in enumerate(state['tasks']):
            cfg = task['config']
            print(str(i + 2) + '/' + str(total_stages) + ':', cfg['name'])
            if task['run_id'] is not None:
                run_dir, result_dir = run_paths(task['run_id'])
                run_manifest = load_batch_json(result_dir / 'run.json') if (result_dir / 'run.json').is_file() else {}
            else:
                run_manifest = {}
            if run_manifest.get('status') != 'trained':
                recoverable = False
                if run_manifest:
                    try:
                        latest_checkpoint(run_dir)
                        recoverable = True
                    except FileNotFoundError:
                        pass
                if not recoverable:
                    task['run_id'] = batch_id + '_' + cfg.get('variant', cfg['rnn']) + '_' + uuid.uuid4().hex[:4]
                    task['attempts'].append(task['run_id'])
                # ID збережено до training: після збою не втратиться.
                atomic_json(ledger, state)
                train_experiment(cfg, run_id=task['run_id'], resume=recoverable)
                _, result_dir = run_paths(task['run_id'])
                if load_batch_json(result_dir / 'run.json')['status'] != 'trained':
                    raise RuntimeError('Навчання перерване; відновіть цей BATCH_ID.')
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            if task['evaluation_dir'] is None:
                _, output = evaluate_experiment(task['run_id'], predict_test=True)
                task['evaluation_dir'] = str(output)
                atomic_json(ledger, state)
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
        print(str(total_stages)+'/'+str(total_stages)+': validation selection + comparison')
        selection_dir = directory / 'selection'
        row = (load_batch_json(selection_dir / 'metrics.json') if (selection_dir / 'metrics.json').is_file()
               else compare_full_models(state, selection_dir))
        if base['use_wandb'] and not state['selection_logged']:
            manifest = dict(name=row['name'], wandb_id=state['selection_wandb_id'],
                wandb_started=state['selection_wandb_started'], provenance=current, versions=runtime_versions())
            weight_key = plan.get('weight_key', 'alpha_lstm')
            selection_cfg = dict(use_wandb=True, kind=kind, architecture='probability_blend_selection',
                source_run_ids=row['members'], **{weight_key: row[weight_key]}, post_rules=row['post_rules'],
                member_configs=[task['config'] for task in state['tasks']], reference_word_LB=reference_lb,
                experiment_batch=batch_id, use_lab=0, selection_policy=row['selection_policy'],
                threshold_candidates=[round(float(t), 2) for t in THRESHOLDS],
                rule_candidates=plan['rule_sets'],
                blend_candidates=plan.get('member_weight_candidates',plan.get('alpha_candidates')))
            wb = open_wandb(selection_cfg, manifest, selection_dir, resume=state['selection_wandb_started'])
            try:
                state['selection_wandb_started'] = True
                atomic_json(ledger, state)
                wb.experiment.summary.update({**selection_cfg, **row})
                state['selection_logged'] = True
                atomic_json(ledger, state)
            finally:
                import wandb
                wandb.finish()
        evaluations = [('baseline', Path(state['baseline_evaluation']))]
        if state.get('reference_selection'):
            evaluations.append(('reference_word_ensemble', Path(state['reference_selection'])))
        evaluations += [('reference_'+item['kind']+'_reported',Path(item['directory']))
                        for item in state.get('reported_references',[])]
        evaluations += [(t['config'].get('variant', t['config']['rnn'] + '_full'), Path(t['evaluation_dir'])) for t in state['tasks']]
        evaluations.append(('validation_selection', selection_dir))
        rows = []
        for candidate, output in evaluations:
            result = load_batch_json(output / 'metrics.json')
            rows.append(dict(candidate=candidate, run_id=result['run_id'],
                **{key: result[key] for key in ('val_f1@0.5', 'val_f1', 'lab_f1', 'thr')},
                submission=str(output / 'submission.csv')))
        comparison = pd.DataFrame(rows).sort_values('val_f1', ascending=False, kind='stable').reset_index(drop=True)
        if study in ('bpe_depth','word_regularization'):
            known = {item['run_id']:item['LB'] for item in state.get('reported_references',[])}
            if state.get('reference_selection') and reference_lb is not None:
                reference_row=load_batch_json(Path(state['reference_selection'])/'metrics.json')
                known[reference_row['run_id']]=reference_lb
            comparison['LB']=[known.get(rid) for rid in comparison['run_id']]
        comparison.to_csv(directory / 'comparison.csv', index=False)
        # При нічиїй — перший кандидат; labeling/LB не впливають на вибір.
        winner = comparison.iloc[0]
        (directory / 'submission_best.csv').write_bytes(Path(winner['submission']).read_bytes())
        winner_record = {k: (None if pd.isna(v) else v) for k, v in winner.to_dict().items()}
        state.update(status='finished', winner=winner_record, selection_dir=str(selection_dir))
        if study == 'bpe_depth':
            # Окрема гіпотеза: BPE може узагальнюватися інакше, ніж показує наш validation.
            # Не підмінюємо цей файл старим word CSV, навіть якщо word виграв на validation.
            (directory / 'submission_bpe_candidate.csv').write_bytes((selection_dir / 'submission.csv').read_bytes())
            state['bpe_candidate'] = row
            print('BPE-кандидат для окремої перевірки:', directory / 'submission_bpe_candidate.csv')
        if study == 'word_regularization':
            candidate_path=directory/'submission_regularized_candidate.csv'
            candidate_path.write_bytes((selection_dir/'submission.csv').read_bytes())
            state['regularized_candidate']=row
            print('Новий regularized кандидат:',candidate_path)
        print(comparison[['candidate', 'val_f1@0.5', 'val_f1', 'lab_f1', 'thr']].to_string(index=False))
        print('Готовий CSV:', directory / 'submission_best.csv', '| автоматичного Kaggle upload немає.')
        return batch_id, comparison
    except BaseException as exc:
        state['status'] = 'interrupted' if isinstance(exc, (KeyboardInterrupt, SystemExit)) else 'failed'
        state['error_type'] = type(exc).__name__
        raise
    finally:
        state['elapsed_seconds'] += time.monotonic() - started
        atomic_json(ledger, state)
        print('Batch time (хв):', round(state['elapsed_seconds'] / 60, 1), '| BATCH_ID:', batch_id)

### 8. W&B login — лише для нового Runtime

In [11]:
# 8. Авторизація потрібна один раз на Runtime; у поточному kernel вона вже готова.
if not globals().get('_WANDB_READY', False):
    login_wandb()
else:
    print('W&B login уже готовий у цьому Runtime.')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mkovalenko (mkovalenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login готовий; доступ до проєкту буде перевірено при створенні run.


### 9. Налаштування довшого запуску

Новий batch: BATCH_ID=None, RESUME_BATCH=False, RUN_BATCH=True. BATCH_EPOCHS — максимум, не гарантована кількість епох. Не змінюйте план при resume.

In [12]:
# 9. Довший word-level експеримент: однаковий довгий режим, потім dropout і AdamW.
BASELINE_RUN_ID = '20261112T150921Z_8b0634b0'
REFERENCE_WORD_BATCH_ID = 'mykyta_20261112T153807Z_bb0085'
REFERENCE_WORD_LB = 0.55325
REPORTED_LB_SCORES = {'mykyta_20261112T194220Z_164805': 0.55182}
BATCH_STUDY = 'word_regularization'
BATCH_EPOCHS = 10        # Максимум; early stopping: min_epochs=3, patience=2, min_delta=0.001.
BATCH_ID = None         # Resume: попередній BATCH_ID цього study.
RESUME_BATCH = False
RUN_BATCH = True       # Встановіть True для реального запуску секції 10.

print('3 word BiLSTM: Adam d=0.3 → Adam d=0.4 → AdamW d=0.4, wd=0.01.')
print('Спільні налаштування: lr=0.001, cosine, максимум', BATCH_EPOCHS, 'епох; labeling поза train.')
print('Повний train, той самий seed/split; top-2 F1 + min-loss checkpoints; RUN_BATCH:',RUN_BATCH)

3 word BiLSTM: Adam d=0.3 → Adam d=0.4 → AdamW d=0.4, wd=0.01.
Спільні налаштування: lr=0.001, cosine, максимум 10 епох; labeling поза train.
Повний train, той самий seed/split; top-2 F1 + min-loss checkpoints; RUN_BATCH: True


### 10. Запуск усієї черги

Запускає три тренування і selection в одній комірці. У W&B очікується 4 нові runs; оновлення LB старих references не створює нових runs.

In [ ]:
if RUN_BATCH:
    BATCH_ID, BATCH_RESULTS = run_experiment_batch(
        BASELINE_RUN_ID, epochs=BATCH_EPOCHS, batch_id=BATCH_ID, resume=RESUME_BATCH,
        study=BATCH_STUDY, reference_batch_id=REFERENCE_WORD_BATCH_ID, reference_lb=REFERENCE_WORD_LB,
        reported_lb_scores=REPORTED_LB_SCORES)
else:
    print('Навчання вимкнено. Для довшої word-черги встановіть RUN_BATCH = True у секції 9.')

Cache: word {'big': 373790, 'val': 2000, 'lab': 26, 'test': 477} | validation SHA256: d8786732a89a5cdd


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


BATCH_ID = mykyta_20261112T204918Z_1c1a39 | відновлення: False | план: 10 епохи на модель


LB,0.55325
LB_source,User Kaggle submissi...
alpha_lstm,0.5
architecture,probability_blend_se...
experiment_batch,mykyta_20261112T1538...
history_val_f1_threshold,0.5
kind,word
lab_f1,0.57895
lab_in_train,0
name,mykyta_full_rnn_sele...
+12,...


LB,0.55182
LB_source,User Kaggle submissi...
alpha_1layer,0
architecture,probability_blend_se...
experiment_batch,mykyta_20261112T1942...
history_val_f1_threshold,0.5
kind,bpe
lab_f1,0.53846
lab_in_train,0
name,mykyta_bpe_selection...
+13,...


INFO:lightning_fabric.utilities.seed:Seed set to 21


... (5 outputs left out)

Evaluation: /content/drive/MyDrive/dl-assignment3/results/mykyta_20261112T204918Z_1c1a39_word_control_d03_adam_7e85/eval_20261112T210631Z_74f4
{'checkpoint_epoch': 5, 'checkpoints_evaluated': 3, 'val_f1@0.5': 0.83495146, 'val_f1': 0.83535393, 'lab_f1': 0.46511628, 'thr': 0.55}


INFO:lightning_fabric.utilities.seed:Seed set to 21


3/5: mykyta_word_d04_adam_10ep
RUN_ID = mykyta_20261112T204918Z_1c1a39_word_d04_adam_ce91 | resume: None | train examples: 373790


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loading `train_dataloader` to estimate number of stepping batches.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bot

Результати: BATCH_RESULTS, results/batches/<BATCH_ID>/comparison.csv і validation_search.csv у selection/. Відомий LB додається лише до відповідного вже submitted reference.

**submission_regularized_candidate.csv** — новий кандидат із цієї черги. **submission_best.csv** — найкращий за validation серед нових та references; він може знову містити старий word-прогноз. Для перевірки нової гіпотези на Kaggle використовуйте regularized candidate. Індивідуальні CSV та checkpoint_scores.csv є у evaluation_dir кожної моделі.

Повні checkpoints: checkpoints/<run_id>/last.ckpt і recovery/last.ckpt. Для resume відновіть setup/визначення/login, вставте BATCH_ID і RESUME_BATCH=True з тим самим планом. Кеш і preprocessing не перебудовуються. Не редагуйте ноутбук під час активного навчання.

History val_f1 залишається F1 при 0.5; summary val_f1 — після threshold tuning. Метрика та її legacy regex збережені. Опис експериментів: rnn_experiment_notes.md.